> ## Prerequisites
>
> This notebook queries Neo4j through a **Unity Catalog HTTP connection** to the
> Neo4j MCP server (deployed on AWS AgentCore). Before running it, create that
> connection by following
> [`MCP-MANUAL-SETUP.md`](../workshop-setup/MCP-MANUAL-SETUP.md), and confirm its
> `url` ends in `/mcp` and `is_mcp_connection` reads `true`.
>
> You also need the Document-Chunk structure from
> [01 Data and Embeddings](01_data_and_embeddings.ipynb) loaded into the graph.

# Querying Neo4j via MCP (Model Context Protocol)

This notebook demonstrates how to query the same Neo4j knowledge graph from
Labs 2-3 through an **MCP server**, reached over a **Unity Catalog HTTP
connection** instead of a direct Neo4j driver.

**What is MCP?** The Model Context Protocol is an open standard for giving AI
tools structured access to data sources. Instead of connecting directly to a
database with credentials, your code talks to an MCP server that manages the
connection.

**How we reach it here:** Unity Catalog holds an HTTP connection that points at
the Neo4j MCP server on AWS AgentCore and handles OAuth2 token refresh. From the
notebook you call the built-in `http_request()` SQL function against that
connection. No driver, no client library, no tokens in the notebook.

**Learning Objectives:**
- Call an MCP server from Databricks SQL with `http_request()`
- Discover available tools (`tools/list`) and retrieve the graph schema (`get_neo4j_schema`)
- Execute Cypher queries via MCP's `read_neo4j_cypher` tool
- Replicate the notebook 02 retrieval patterns: document context, adjacent chunks, topology traversal, operating limits
- Understand how a Unity Catalog connection decouples clients from database credentials

---

## Architecture

```
Direct (notebooks 04/05):
  Notebook ──> neo4j driver ──> Neo4j Aura
               (credentials)

MCP via Unity Catalog (this notebook):
  Notebook ──> http_request() ──> UC HTTP Connection ──> AgentCore Gateway ──> Neo4j Aura
               (SQL function)     (OAuth2 M2M, managed)
```

The Unity Catalog connection authenticates to the MCP server with an OAuth2
machine-to-machine flow and refreshes the token automatically. Your notebook
only needs the connection name.

## Configuration

The only thing the notebook needs is the **name** of the Unity Catalog HTTP
connection. Everything else (host, OAuth client id and secret, scope, token
endpoint) lives inside the connection, created in
[`MCP-MANUAL-SETUP.md`](../workshop-setup/MCP-MANUAL-SETUP.md).

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

# Name of the Unity Catalog HTTP connection that points at the Neo4j MCP server.
# Created via workshop-setup/MCP-MANUAL-SETUP.md.
MCP_SERVER = "aircraft_mcp_server"

# The AgentCore Gateway prefixes every tool name with its target id.
TOOL_GET_SCHEMA = "neo4j-mcp-server-target___get_neo4j_schema"
TOOL_READ_CYPHER = "neo4j-mcp-server-target___read_neo4j_cypher"

print(f"Using MCP connection: {MCP_SERVER}")

## Calling the MCP server

Databricks talks to the connection with the built-in
[`http_request()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/http_request)
SQL function. MCP speaks JSON-RPC over HTTP POST, so every call is a single SQL
statement that posts a JSON-RPC body and returns
`STRUCT<status_code INT, text STRING>`.

The simplest possible call, a `tools/list`, looks like this in pure SQL:

```sql
SELECT http_request(
  conn => 'aircraft_mcp_server',
  method => 'POST',
  path => '',
  headers => map('Content-Type', 'application/json'),
  json => '{"jsonrpc":"2.0","method":"tools/list","id":1}'
);
```

The helpers below wrap that pattern. They build the JSON-RPC body, bind it as a
query parameter so quotes and newlines in Cypher never need escaping, send it
through the connection, and parse the response.

In [ ]:
import json


def _http_request(payload_json: str) -> tuple[int, str]:
    """POST a JSON-RPC body to the MCP server through the UC HTTP connection.

    The connection name is a SQL string constant; the body binds as a parameter
    so quotes and newlines inside Cypher need no escaping.
    """
    row = spark.sql(
        f"""
        SELECT http_request(
          conn => '{MCP_SERVER}',
          method => 'POST',
          path => '',
          headers => map('Content-Type', 'application/json'),
          json => :payload
        ) AS response
        """,
        args={"payload": payload_json},
    ).collect()[0]["response"]
    return row["status_code"], row["text"]


def _parse_jsonrpc(text: str) -> dict:
    """Parse a JSON-RPC response, tolerating a server-sent-event wrapper."""
    for line in text.splitlines():
        stripped = line.strip()
        if stripped.startswith("data:"):
            text = stripped[len("data:"):].strip()
            break
    return json.loads(text)


def mcp_rpc(method: str, params: dict | None = None) -> dict:
    """Send a JSON-RPC request and return its `result` object."""
    payload = json.dumps(
        {"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}
    )
    status, text = _http_request(payload)
    if status != 200:
        raise RuntimeError(f"HTTP {status}: {text}")
    body = _parse_jsonrpc(text)
    if "error" in body:
        raise RuntimeError(f"MCP error: {body['error']}")
    return body.get("result", {})


def mcp_call_tool(tool_name: str, arguments: dict | None = None) -> str:
    """Call an MCP tool and return its concatenated text payload."""
    result = mcp_rpc("tools/call", {"name": tool_name, "arguments": arguments or {}})
    return "\n".join(part.get("text", "") for part in result.get("content", []))


def read_cypher(query: str) -> str:
    """Run a Cypher query through the MCP read_neo4j_cypher tool."""
    return mcp_call_tool(TOOL_READ_CYPHER, {"query": query})


print("MCP helpers ready: mcp_rpc, mcp_call_tool, read_cypher")

---

# Part 1: Discover and Explore

Every MCP server exposes a set of **tools** that clients can discover and call. This is the MCP equivalent of exploring an API's endpoints.

## List Available Tools

Discover what tools the MCP server provides. This `tools/list` call also doubles as a connectivity check.

In [ ]:
result = mcp_rpc("tools/list")
tools = result.get("tools", [])
print(f"Available tools ({len(tools)}):\n")
for tool in tools:
    print(f"  {tool['name']}")
    description = (tool.get("description") or "").strip().splitlines()
    if description:
        print(f"    {description[0][:100]}")
    print()

## Get the Graph Schema

The `get_neo4j_schema` tool returns node labels, relationship types, and
properties. It is the most basic request you can make against the server. Tool
names are prefixed by the AgentCore Gateway, so the call uses
`neo4j-mcp-server-target___get_neo4j_schema`.

First, the raw SQL form: a single `http_request()` against the connection.

In [ ]:
# The simplest possible call: a basic get_neo4j_schema in one SQL statement.
schema_payload = json.dumps({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {"name": TOOL_GET_SCHEMA, "arguments": {}},
})

display(spark.sql(
    f"""
    SELECT http_request(
      conn => '{MCP_SERVER}',
      method => 'POST',
      path => '',
      headers => map('Content-Type', 'application/json'),
      json => :payload
    ) AS response
    """,
    args={"payload": schema_payload},
))

The `response` column is the `STRUCT<status_code, text>` returned by
`http_request()`. The helpers do exactly this, then pull the schema out of
`text` for you.

In [ ]:
schema = mcp_call_tool(TOOL_GET_SCHEMA)
print("Graph Schema:")
print("=" * 70)
print(schema)

## Graph Statistics

Run a Cypher query through MCP to count nodes by label.

In [ ]:
result = read_cypher("""
        MATCH (n)
        WITH labels(n) AS nodeLabels
        UNWIND nodeLabels AS label
        RETURN label, count(*) AS count
        ORDER BY count DESC
""")
print('Node counts by label:\n')
print(result)

## Aircraft Fleet Overview

Query the aircraft topology that was loaded in Lab 2.

In [ ]:
result = read_cypher("""
        MATCH (a:Aircraft)-[:HAS_SYSTEM]->(s:System)
        RETURN a.tail_number AS aircraft, a.model AS model,
               collect(s.name) AS systems
        ORDER BY a.tail_number
        LIMIT 5
""")
print('Aircraft Fleet:\n')
print(result)

---

# Part 2: Document-Chunk Queries

In notebook 01, we created a Document-Chunk structure from maintenance manuals. These queries explore that structure through MCP.

## Document Overview

List documents and their chunk counts.

In [ ]:
result = read_cypher("""
        MATCH (d:Document)
        OPTIONAL MATCH (d)<-[:FROM_DOCUMENT]-(c:Chunk)
        RETURN d.documentId AS document_id, d.title AS title,
               d.aircraftType AS aircraft_type, count(c) AS chunks
""")
print('Documents in the graph:\n')
print(result)

## Chunk Chain Traversal

Walk the `NEXT_CHUNK` relationships created in notebook 01 to see how chunks are linked sequentially.

In [ ]:
result = read_cypher("""
        MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document)
        WHERE c.index IS NOT NULL
        OPTIONAL MATCH (c)-[:NEXT_CHUNK]->(next:Chunk)
        RETURN c.index AS chunk_index,
               substring(c.text, 0, 80) AS preview,
               next.index AS next_chunk
        ORDER BY c.index
        LIMIT 5
""")
print('Chunk chain (first 5):\n')
print(result)

---

# Part 3: Graph-Enhanced Search via MCP

In notebook 02, the `VectorCypherRetriever` combined vector search with graph traversal behind a Python abstraction. Here you write the equivalent Cypher explicitly and send it through MCP's `read_neo4j_cypher` tool.

We use **fulltext search** (`db.index.fulltext.queryNodes`) as the entry point instead of vector search, since the MCP client doesn't need an embedding model.

> **Note:** The `maintenanceChunkText` fulltext index was created in notebook 01.

## Example 1: Document Context Enrichment

Find relevant chunks via fulltext search, then traverse `FROM_DOCUMENT` to get source document metadata.

This mirrors notebook 02's `document_context_query` VectorCypherRetriever.

In [ ]:
result = read_cypher("""
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'engine vibration troubleshoot')
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        RETURN doc.documentId AS document_id,
               doc.aircraftType AS aircraft_type,
               doc.title AS title,
               node.index AS chunk_index,
               score,
               substring(node.text, 0, 200) AS context
        ORDER BY score DESC
        LIMIT 3
""")
print('Document-enriched search results:\n')
print(result)

**How it works:**
1. Fulltext search finds chunks matching the keywords
2. Graph traversal follows `FROM_DOCUMENT` to get document metadata
3. Results include document ID, aircraft type, title alongside the text

Compare to notebook 02's VectorCypherRetriever which does the same traversal but uses vector similarity (embeddings) as the entry point instead of keyword matching.

## Example 2: Adjacent Chunk Retrieval

Retrieve surrounding context by following `NEXT_CHUNK` relationships forward and backward.

This mirrors notebook 02's `adjacent_chunks_query` VectorCypherRetriever.

In [ ]:
result = read_cypher("""
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'hydraulic pressure limits')
        YIELD node, score
        OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
        OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        RETURN doc.documentId AS document_id,
               node.index AS chunk_index,
               substring(COALESCE(prev.text, ''), 0, 100) AS previous_context,
               substring(node.text, 0, 200) AS main_context,
               substring(COALESCE(next.text, ''), 0, 100) AS next_context
        ORDER BY score DESC
        LIMIT 3
""")
print('Adjacent chunk retrieval:\n')
print(result)

**Why this matters:** Maintenance procedures often span multiple chunks. By retrieving the previous and next chunks alongside the match, you get more complete procedure context. This is exactly what the `VectorCypherRetriever` does under the hood in notebook 02.

## Example 3: Aircraft Topology Traversal

Start from a search result, traverse through Document -> Aircraft -> System -> Component.

This mirrors notebook 02's `system_context_query` VectorCypherRetriever and demonstrates the full power of graph-enhanced retrieval.

In [ ]:
result = read_cypher("""
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'fuel pump maintenance')
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)-[:APPLIES_TO]->(a:Aircraft)
        MATCH (a)-[:HAS_SYSTEM]->(s:System)
        OPTIONAL MATCH (s)-[:HAS_COMPONENT]->(comp:Component)
        WITH node, doc, a, s, comp, score
        RETURN doc.documentId AS document_id,
               doc.aircraftType AS aircraft_type,
               a.tail_number AS aircraft,
               COLLECT(DISTINCT s.name)[0..3] AS systems,
               COLLECT(DISTINCT comp.name)[0..3] AS components,
               substring(node.text, 0, 200) AS context,
               score
        ORDER BY score DESC
        LIMIT 3
""")
print('Topology-enriched search:\n')
print(result)

**Graph traversal path:**
1. Fulltext search finds relevant maintenance chunks
2. `FROM_DOCUMENT` reaches the source Document
3. `APPLIES_TO` connects to the Aircraft
4. `HAS_SYSTEM` and `HAS_COMPONENT` walk the topology hierarchy

The result combines text context with structured aircraft data (systems, components) from a single query.

## Example 4: Operating Limits from Extracted Entities

Traverse the full chain: Chunk -> Document -> Aircraft -> System -> Sensor -> OperatingLimit.

In notebook 01, `SimpleKGPipeline` extracted `OperatingLimit` entities from maintenance text and cross-linked them to Sensor nodes. This query reaches those structured entities through graph traversal.

This mirrors notebook 02's `operating_limit_query` VectorCypherRetriever.

In [ ]:
result = read_cypher("""
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'EGT temperature limits')
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)-[:APPLIES_TO]->(a:Aircraft)
        OPTIONAL MATCH (a)-[:HAS_SYSTEM]->(sys:System)-[:HAS_SENSOR]->(s:Sensor)-[:HAS_LIMIT]->(ol:OperatingLimit)
        WITH node, doc, a, score,
             COLLECT(DISTINCT {
                 sensor: s.type,
                 parameter: ol.parameterName,
                 max: ol.maxValue,
                 unit: ol.unit,
                 regime: ol.regime
             })[0..5] AS operating_limits
        RETURN doc.aircraftType AS aircraft_type,
               operating_limits,
               substring(node.text, 0, 200) AS context
        ORDER BY score DESC
        LIMIT 3
""")
print('Operating limits from graph traversal:\n')
print(result)

**Full traversal chain:** Chunk -> Document -> Aircraft -> System -> Sensor -> OperatingLimit

This demonstrates the entity extraction payoff from notebook 01: the LLM extracted structured `OperatingLimit` entities from unstructured text, and those entities are now queryable through graph traversal via MCP.

> **Note:** The number of OperatingLimit entities depends on what the LLM extracted during the SimpleKGPipeline run in notebook 01. If entity extraction produced no OperatingLimit entities, the `operating_limits` field will be empty.

---

# Try Your Own Queries

Experiment with different Cypher queries through MCP. Some ideas:

- What are the vibration limits that require engine shutdown?
- How do I check for hydraulic fluid contamination?
- What oil analysis levels indicate bearing wear?
- When should I perform a borescope inspection?

In [ ]:
# Try your own query
result = read_cypher("""
        CALL db.index.fulltext.queryNodes('maintenanceChunkText', 'YOUR SEARCH TERMS HERE')
        YIELD node, score
        RETURN score, substring(node.text, 0, 300) AS context
        ORDER BY score DESC
        LIMIT 3
""")
print(result)

---

# Comparing Approaches

| Aspect | Direct Driver (notebooks 04/05) | MCP via Unity Catalog (this notebook) |
|--------|------------------------|-------------|
| Connection | `neo4j://` driver with credentials | UC HTTP connection + `http_request()` |
| Authentication | Neo4j username/password | OAuth2 machine-to-machine, managed by Unity Catalog |
| Dependencies | `neo4j`, `neo4j-graphrag` | None (built-in SQL function) |
| Search Entry Point | Vector search (embeddings) | Fulltext search (keywords) |
| Graph Traversal | `VectorCypherRetriever` abstraction | Explicit Cypher via `read_neo4j_cypher` |
| Best For | Application code, GraphRAG pipelines | AI agents, tool-calling, governed access |

## Summary

In this notebook, you queried the same Neo4j knowledge graph from Labs 2-3 entirely through MCP, reached over a Unity Catalog HTTP connection:

**Part 1 - Discover & Explore:**
- Called an MCP server from Databricks SQL with `http_request()`, with no driver or client library
- Discovered tools (`get_neo4j_schema`, `read_neo4j_cypher`) and retrieved the graph schema
- Ran basic Cypher queries for node counts and fleet overview

**Part 2 - Document-Chunk Queries:**
- Explored the Document-Chunk structure created in notebook 01
- Walked `NEXT_CHUNK` chains to see sequential chunk linking

**Part 3 - Graph-Enhanced Search:**
- Document context enrichment (fulltext search + `FROM_DOCUMENT`)
- Adjacent chunk retrieval (fulltext search + `NEXT_CHUNK`)
- Aircraft topology traversal (fulltext search + `APPLIES_TO` -> `HAS_SYSTEM` -> `HAS_COMPONENT`)
- Operating limits via full entity chain traversal

**Key takeaway:** A Unity Catalog HTTP connection puts a governed, credential-free layer between your code and the MCP server. The Cypher queries are identical to what runs inside the `VectorCypherRetriever` abstractions from notebook 02. The connection just changes how they get to Neo4j.